# Multi-Objective Readout Policy Optimization (Colab)

Notebook 13 is the capstone for the readout-side notebooks.

Earlier notebooks:

```text
08 -> deterministic readout scheduling
09 -> adaptive early stopping
10 -> confidence-aware scheduling
11 -> hybrid modwheel + confidence lane coverage
12 -> coverage-constrained stopping policy
```

Notebook 13 turns the policy choice into a multi-objective optimization problem:

```text
minimize query cost
maximize predictive behavior
maximize lane coverage
minimize class-distribution shift
```

We evaluate schedules across progressive fractions, compute objective scores, and identify Pareto-efficient readout policies.

Guardrail: this notebook evaluates classical readout policy selection. It does **not** claim QOS improvement, quantum advantage, or model accuracy improvement.


In [ ]:
# Clone your fork and move into repo root.
# If Colab says the folder already exists, restart runtime or comment these two lines after first run.
!git clone https://github.com/thinkthoughts/quantum-oracle-sketching-mod30.git
%cd quantum-oracle-sketching-mod30


In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from modwheel import STANDARD_WHEELS

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score

fig_dir = Path("figures")
data_dir = Path("data")
fig_dir.mkdir(exist_ok=True)
data_dir.mkdir(exist_ok=True)


## 1. Load 20news and train baseline classifier

In [ ]:
RANDOM_STATE = 9423
N_RANDOM_TRIALS = 30

categories = [
    "comp.graphics",
    "rec.sport.baseball",
    "sci.space",
    "talk.politics.misc",
]

dataset = fetch_20newsgroups(
    subset="all",
    categories=categories,
    remove=("headers", "footers", "quotes"),
    shuffle=True,
    random_state=RANDOM_STATE,
)

texts = np.array(dataset.data, dtype=object)
y = np.array(dataset.target)
target_names = dataset.target_names

texts_train, texts_test, y_train, y_test = train_test_split(
    texts,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

vectorizer = TfidfVectorizer(max_features=12000, min_df=2, stop_words="english")
X_train = vectorizer.fit_transform(texts_train)
X_test = vectorizer.transform(texts_test)

clf = LinearSVC(random_state=RANDOM_STATE, dual="auto")
t0 = time.perf_counter()
clf.fit(X_train, y_train)
train_time = time.perf_counter() - t0

full_pred = clf.predict(X_test)
full_accuracy = accuracy_score(y_test, full_pred)
full_balanced_accuracy = balanced_accuracy_score(y_test, full_pred)

baseline = {
    "n_train": X_train.shape[0],
    "n_test": X_test.shape[0],
    "n_features": X_train.shape[1],
    "train_time_seconds": train_time,
    "full_accuracy": full_accuracy,
    "full_balanced_accuracy": full_balanced_accuracy,
}
baseline


## 2. Confidence, coverage, and evaluation helpers

In [ ]:
def decision_margin_confidence(model, X):
    scores = model.decision_function(X)
    if scores.ndim == 1:
        return np.abs(scores)
    sorted_scores = np.sort(scores, axis=1)
    return sorted_scores[:, -1] - sorted_scores[:, -2]


confidence = decision_margin_confidence(clf, X_test)
all_idx = np.arange(X_test.shape[0])


def class_balance_l1_shift(y_subset, y_reference):
    n_classes = len(np.unique(y_reference))
    subset_counts = np.bincount(y_subset, minlength=n_classes)
    ref_counts = np.bincount(y_reference, minlength=n_classes)
    subset_frac = subset_counts / subset_counts.sum()
    ref_frac = ref_counts / ref_counts.sum()
    return float(np.sum(np.abs(subset_frac - ref_frac)))


def lane_coverage_fraction(indices, wheel_name="mod30"):
    wheel = STANDARD_WHEELS[wheel_name]
    residues = set(wheel.residues)
    covered = set([int(i % wheel.modulus) for i in indices if int(i % wheel.modulus) in residues])
    return len(covered) / len(residues)


def evaluate_indices(indices):
    pred = clf.predict(X_test[indices])
    yt = y_test[indices]
    return {
        "accuracy": accuracy_score(yt, pred),
        "balanced_accuracy": balanced_accuracy_score(yt, pred),
        "class_balance_l1_shift": class_balance_l1_shift(yt, y_test),
        "mean_confidence": float(np.mean(confidence[indices])),
        "median_confidence": float(np.median(confidence[indices])),
        "lane_coverage_fraction": lane_coverage_fraction(indices, "mod30"),
    }


confidence_summary = {
    "min": float(np.min(confidence)),
    "median": float(np.median(confidence)),
    "mean": float(np.mean(confidence)),
    "max": float(np.max(confidence)),
}
confidence_summary


## 3. Schedule constructors

In [ ]:
def wheel_query_indices(n_queries, wheel_name="mod30"):
    wheel = STANDARD_WHEELS[wheel_name]
    residues = set(wheel.residues)
    return np.array([i for i in range(n_queries) if i % wheel.modulus in residues], dtype=int)


def random_schedule(n_keep, seed):
    rng = np.random.default_rng(seed)
    return np.array(rng.choice(all_idx, size=n_keep, replace=False), dtype=int)


def global_low_confidence_schedule(n_keep):
    return all_idx[np.argsort(confidence[all_idx])][:n_keep]


def global_high_confidence_schedule(n_keep):
    return all_idx[np.argsort(-confidence[all_idx])][:n_keep]


def mod_lanes(indices, wheel_name="mod30"):
    wheel = STANDARD_WHEELS[wheel_name]
    lanes = {r: [] for r in list(wheel.residues)}
    for i in indices:
        r = int(i % wheel.modulus)
        if r in lanes:
            lanes[r].append(int(i))
    return lanes


def interleave_lanes(lanes):
    output = []
    keys = list(lanes.keys())
    max_len = max(len(v) for v in lanes.values() if len(v) > 0)
    for j in range(max_len):
        for k in keys:
            if j < len(lanes[k]):
                output.append(lanes[k][j])
    return np.array(output, dtype=int)


def hybrid_lane_confidence_schedule(n_keep, wheel_name="mod30", confidence_order="low"):
    base = wheel_query_indices(X_test.shape[0], wheel_name)
    lanes = mod_lanes(base, wheel_name=wheel_name)

    for r, values in lanes.items():
        arr = np.array(values, dtype=int)
        if len(arr) == 0:
            lanes[r] = []
            continue
        if confidence_order == "low":
            arr = arr[np.argsort(confidence[arr])]
        elif confidence_order == "high":
            arr = arr[np.argsort(-confidence[arr])]
        else:
            raise ValueError("confidence_order must be 'low' or 'high'")
        lanes[r] = arr.tolist()

    return interleave_lanes(lanes)[:n_keep]


def hybrid_lane_round_robin_schedule(n_keep, wheel_name="mod30"):
    base = wheel_query_indices(X_test.shape[0], wheel_name)
    lanes = mod_lanes(base, wheel_name=wheel_name)
    return interleave_lanes(lanes)[:n_keep]


mod30_idx = wheel_query_indices(X_test.shape[0], "mod30")
n_keep = len(mod30_idx)

base_schedules = {
    "mod30": mod30_idx,
    "global_low_conf": global_low_confidence_schedule(n_keep),
    "global_high_conf": global_high_confidence_schedule(n_keep),
    "hybrid_lane_low_conf": hybrid_lane_confidence_schedule(n_keep, "mod30", "low"),
    "hybrid_lane_high_conf": hybrid_lane_confidence_schedule(n_keep, "mod30", "high"),
    "hybrid_lane_round_robin": hybrid_lane_round_robin_schedule(n_keep, "mod30"),
}

{k: (len(v), lane_coverage_fraction(v)) for k, v in base_schedules.items()}


## 4. Generate candidate policies

In [ ]:
fractions = np.linspace(0.05, 1.0, 20)

def progressive_policy_points(schedule_indices, fractions, schedule_name, schedule_type, trial=-1):
    rows = []
    total_queries = X_test.shape[0]
    for frac in fractions:
        k = max(2, int(np.ceil(len(schedule_indices) * frac)))
        idx = schedule_indices[:k]
        t0 = time.perf_counter()
        metrics = evaluate_indices(idx)
        eval_time = time.perf_counter() - t0

        rows.append({
            "schedule_name": schedule_name,
            "schedule_type": schedule_type,
            "trial": trial,
            "schedule_fraction": frac,
            "n_eval": len(idx),
            "fraction_of_all_queries": len(idx) / total_queries,
            "eval_time_seconds": eval_time,
            **metrics,
        })
    return pd.DataFrame(rows)


policy_rows = []
schedule_types = {
    "mod30": "deterministic_mod30",
    "global_low_conf": "global_low_confidence",
    "global_high_conf": "global_high_confidence",
    "hybrid_lane_low_conf": "hybrid_lane_low_confidence",
    "hybrid_lane_high_conf": "hybrid_lane_high_confidence",
    "hybrid_lane_round_robin": "hybrid_lane_round_robin",
}

for name, idx in base_schedules.items():
    policy_rows.append(progressive_policy_points(idx, fractions, name, schedule_types[name], trial=-1))

for trial in range(N_RANDOM_TRIALS):
    ridx = random_schedule(n_keep, seed=RANDOM_STATE + 13000 + trial * 1009)
    policy_rows.append(progressive_policy_points(ridx, fractions, "random_matched", "random_matched", trial=trial))

policies_df = pd.concat(policy_rows, ignore_index=True)
policies_csv = data_dir / "13_readout_policy_candidates.csv"
policies_df.to_csv(policies_csv, index=False)

policies_df.head()


## 5. Normalize objectives and compute weighted scores

In [ ]:
# Objective directions:
# - smaller fraction_of_all_queries is better
# - larger balanced_accuracy is better
# - larger lane_coverage_fraction is better
# - smaller class_balance_l1_shift is better

def add_objective_columns(df):
    out = df.copy()

    # Normalize all components to [0, 1]-like cost terms where lower is better.
    out["cost_query_fraction"] = out["fraction_of_all_queries"]
    out["cost_accuracy_gap"] = np.maximum(0, full_balanced_accuracy - out["balanced_accuracy"]) / max(full_balanced_accuracy, 1e-9)
    out["cost_coverage_gap"] = 1 - out["lane_coverage_fraction"]

    max_shift = max(out["class_balance_l1_shift"].max(), 1e-9)
    out["cost_class_shift"] = out["class_balance_l1_shift"] / max_shift

    return out


policies_obj = add_objective_columns(policies_df)

weight_sets = {
    "balanced": {
        "query": 0.25,
        "accuracy": 0.35,
        "coverage": 0.25,
        "class_shift": 0.15,
    },
    "min_queries": {
        "query": 0.55,
        "accuracy": 0.25,
        "coverage": 0.10,
        "class_shift": 0.10,
    },
    "accuracy_first": {
        "query": 0.15,
        "accuracy": 0.60,
        "coverage": 0.15,
        "class_shift": 0.10,
    },
    "coverage_first": {
        "query": 0.15,
        "accuracy": 0.25,
        "coverage": 0.50,
        "class_shift": 0.10,
    },
    "distribution_first": {
        "query": 0.15,
        "accuracy": 0.25,
        "coverage": 0.20,
        "class_shift": 0.40,
    },
}


def score_policies(df, weights):
    return (
        weights["query"] * df["cost_query_fraction"]
        + weights["accuracy"] * df["cost_accuracy_gap"]
        + weights["coverage"] * df["cost_coverage_gap"]
        + weights["class_shift"] * df["cost_class_shift"]
    )


for name, weights in weight_sets.items():
    policies_obj[f"score_{name}"] = score_policies(policies_obj, weights)

policies_obj_csv = data_dir / "13_readout_policy_objectives.csv"
policies_obj.to_csv(policies_obj_csv, index=False)

policies_obj.head()


## 6. Pareto frontier

In [ ]:
# Pareto efficiency for objectives:
# minimize query fraction, minimize accuracy gap, minimize coverage gap, minimize class shift.

objective_cols = [
    "cost_query_fraction",
    "cost_accuracy_gap",
    "cost_coverage_gap",
    "cost_class_shift",
]

def pareto_efficient(costs):
    n = costs.shape[0]
    is_eff = np.ones(n, dtype=bool)
    for i in range(n):
        if not is_eff[i]:
            continue
        # A point is dominated if another point is <= in all costs and < in at least one.
        dominates_i = np.all(costs <= costs[i], axis=1) & np.any(costs < costs[i], axis=1)
        if np.any(dominates_i):
            is_eff[i] = False
    return is_eff


cost_matrix = policies_obj[objective_cols].to_numpy()
policies_obj["pareto_efficient"] = pareto_efficient(cost_matrix)

pareto_df = policies_obj[policies_obj["pareto_efficient"]].copy()
pareto_csv = data_dir / "13_readout_policy_pareto_frontier.csv"
pareto_df.to_csv(pareto_csv, index=False)

len(policies_obj), len(pareto_df), pareto_df[["schedule_name", "schedule_type", "fraction_of_all_queries", "balanced_accuracy", "lane_coverage_fraction", "class_balance_l1_shift"]].head()


## 7. Best policy by objective weights

In [ ]:
best_rows = []
for name in weight_sets:
    score_col = f"score_{name}"
    best = policies_obj.sort_values(score_col).iloc[0]
    best_rows.append({
        "weight_set": name,
        "best_schedule_name": best["schedule_name"],
        "best_schedule_type": best["schedule_type"],
        "best_trial": int(best["trial"]),
        "best_score": float(best[score_col]),
        "fraction_of_all_queries": float(best["fraction_of_all_queries"]),
        "n_eval": int(best["n_eval"]),
        "balanced_accuracy": float(best["balanced_accuracy"]),
        "accuracy": float(best["accuracy"]),
        "lane_coverage_fraction": float(best["lane_coverage_fraction"]),
        "class_balance_l1_shift": float(best["class_balance_l1_shift"]),
        "mean_confidence": float(best["mean_confidence"]),
        "pareto_efficient": bool(best["pareto_efficient"]),
    })

best_policy_df = pd.DataFrame(best_rows)
best_policy_csv = data_dir / "13_best_policy_by_weights.csv"
best_policy_df.to_csv(best_policy_csv, index=False)
best_policy_df


## 8. Figure 13a — accuracy vs query fraction with Pareto points

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.6))

# All candidate policies faint.
ax.scatter(
    policies_obj["fraction_of_all_queries"],
    policies_obj["balanced_accuracy"],
    alpha=0.25,
    label="candidate policies",
)

# Pareto front highlighted.
ax.scatter(
    pareto_df["fraction_of_all_queries"],
    pareto_df["balanced_accuracy"],
    marker="D",
    s=60,
    label="Pareto-efficient",
)

ax.axhline(full_balanced_accuracy, linestyle="--", linewidth=1, label="full readout")
ax.axhline(0.95 * full_balanced_accuracy, linestyle=":", linewidth=1, label="95% target")
ax.set_title("Readout Policy Space: Accuracy vs Query Cost")
ax.set_xlabel("Fraction of all test queries evaluated")
ax.set_ylabel("Balanced accuracy")
ax.set_ylim(0.5, 1.0)
ax.grid(True, alpha=0.35)
ax.legend()
fig.tight_layout()

fig_a_svg = fig_dir / "figure_13a_accuracy_vs_queries_pareto.svg"
fig_a_png = fig_dir / "figure_13a_accuracy_vs_queries_pareto.png"
fig.savefig(fig_a_svg)
fig.savefig(fig_a_png, dpi=220)
plt.show()
fig_a_svg, fig_a_png


## 9. Figure 13b — coverage vs query fraction

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.6))

for stype, label in [
    ("deterministic_mod30", "mod30"),
    ("global_low_confidence", "global low-conf"),
    ("hybrid_lane_low_confidence", "hybrid lanes + low-conf"),
    ("hybrid_lane_round_robin", "hybrid lane round-robin"),
]:
    df = policies_obj[policies_obj["schedule_type"] == stype].sort_values("fraction_of_all_queries")
    ax.plot(df["fraction_of_all_queries"], df["lane_coverage_fraction"], marker="o", linewidth=2, label=label)

random_mean = (
    policies_obj[policies_obj["schedule_type"] == "random_matched"]
    .groupby("fraction_of_all_queries")["lane_coverage_fraction"]
    .mean()
    .reset_index()
)
ax.plot(random_mean["fraction_of_all_queries"], random_mean["lane_coverage_fraction"], linewidth=2, label="random mean")

ax.set_title("Lane Coverage vs Query Cost")
ax.set_xlabel("Fraction of all test queries evaluated")
ax.set_ylabel("Mod30 lane coverage fraction")
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.35)
ax.legend()
fig.tight_layout()

fig_b_svg = fig_dir / "figure_13b_coverage_vs_queries.svg"
fig_b_png = fig_dir / "figure_13b_coverage_vs_queries.png"
fig.savefig(fig_b_svg)
fig.savefig(fig_b_png, dpi=220)
plt.show()
fig_b_svg, fig_b_png


## 10. Figure 13c — Pareto frontier colored by coverage

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.6))

sc = ax.scatter(
    policies_obj["fraction_of_all_queries"],
    policies_obj["balanced_accuracy"],
    c=policies_obj["lane_coverage_fraction"],
    alpha=0.45,
)

ax.scatter(
    pareto_df["fraction_of_all_queries"],
    pareto_df["balanced_accuracy"],
    marker="D",
    s=70,
    edgecolors="black",
    facecolors="none",
    label="Pareto-efficient",
)

ax.axhline(0.95 * full_balanced_accuracy, linestyle=":", linewidth=1, label="95% target")
ax.set_title("Pareto View: Accuracy, Query Cost, and Lane Coverage")
ax.set_xlabel("Fraction of all test queries evaluated")
ax.set_ylabel("Balanced accuracy")
ax.set_ylim(0.5, 1.0)
ax.grid(True, alpha=0.35)
ax.legend()
fig.colorbar(sc, ax=ax, label="Lane coverage fraction")
fig.tight_layout()

fig_c_svg = fig_dir / "figure_13c_pareto_coverage_view.svg"
fig_c_png = fig_dir / "figure_13c_pareto_coverage_view.png"
fig.savefig(fig_c_svg)
fig.savefig(fig_c_png, dpi=220)
plt.show()
fig_c_svg, fig_c_png


## 11. Figure 13d — best policy by weight set

In [ ]:
fig, ax = plt.subplots(figsize=(10.0, 5.8))

x = np.arange(len(best_policy_df))
ax.bar(x, best_policy_df["fraction_of_all_queries"])
ax.set_title("Best Policy by Objective Weight Set")
ax.set_xlabel("Objective weighting")
ax.set_ylabel("Fraction of all queries evaluated")
ax.set_xticks(x)
ax.set_xticklabels(best_policy_df["weight_set"], rotation=20)
ax.grid(True, axis="y", alpha=0.35)

for i, row in best_policy_df.iterrows():
    label = row["best_schedule_name"]
    ax.text(i, row["fraction_of_all_queries"], f"{label}\n{row['balanced_accuracy']:.3f}", ha="center", va="bottom", fontsize=9)

fig.tight_layout()

fig_d_svg = fig_dir / "figure_13d_best_policy_by_weights.svg"
fig_d_png = fig_dir / "figure_13d_best_policy_by_weights.png"
fig.savefig(fig_d_svg)
fig.savefig(fig_d_png, dpi=220)
plt.show()
fig_d_svg, fig_d_png


## 12. Figure 13e — score heatmap

In [ ]:
score_cols = [f"score_{name}" for name in weight_sets]
compact = policies_obj.copy()

# For readability, use best score per schedule_type for each weight set.
heat_rows = []
for stype, group in compact.groupby("schedule_type"):
    row = {"schedule_type": stype}
    for score_col in score_cols:
        row[score_col] = group[score_col].min()
    heat_rows.append(row)

heat_df = pd.DataFrame(heat_rows).set_index("schedule_type")
heat_df = heat_df[score_cols]

fig, ax = plt.subplots(figsize=(10.0, 5.8))
im = ax.imshow(heat_df.values, aspect="auto")
ax.set_title("Minimum Objective Score by Schedule Type")
ax.set_xticks(np.arange(len(score_cols)))
ax.set_xticklabels([c.replace("score_", "") for c in score_cols], rotation=20)
ax.set_yticks(np.arange(len(heat_df.index)))
ax.set_yticklabels(heat_df.index)

for i in range(heat_df.shape[0]):
    for j in range(heat_df.shape[1]):
        ax.text(j, i, f"{heat_df.values[i, j]:.3f}", ha="center", va="center", fontsize=8)

fig.colorbar(im, ax=ax, label="Objective score (lower better)")
fig.tight_layout()

fig_e_svg = fig_dir / "figure_13e_score_heatmap.svg"
fig_e_png = fig_dir / "figure_13e_score_heatmap.png"
fig.savefig(fig_e_svg)
fig.savefig(fig_e_png, dpi=220)
plt.show()
fig_e_svg, fig_e_png


## 13. Interpretation helper

In [ ]:
print("Full balanced accuracy:", full_balanced_accuracy)
print("Pareto-efficient policies:", len(pareto_df), "of", len(policies_obj))
print()
display(best_policy_df)

print("\nBest policies:")
for _, row in best_policy_df.iterrows():
    print(
        f"{row['weight_set']}: {row['best_schedule_name']} "
        f"({row['best_schedule_type']}), query_fraction={row['fraction_of_all_queries']:.3f}, "
        f"bal_acc={row['balanced_accuracy']:.3f}, coverage={row['lane_coverage_fraction']:.3f}, "
        f"class_shift={row['class_balance_l1_shift']:.3f}, pareto={row['pareto_efficient']}"
    )

print("""
Notebook 13 interpretation:

- Notebook 12 defined fixed stopping policies.
- Notebook 13 searches across policy candidates and objective weights.
- Pareto-efficient policies represent tradeoffs where no objective can improve without worsening another.
- Different weight sets choose different policies, making the readout policy decision explicit.
- The value is policy selection and tradeoff analysis, not accuracy improvement.

Guardrail:
This is a classical multi-objective readout policy analysis, not a QOS improvement or quantum advantage claim.
""")


## 14. Download outputs

In [ ]:
from google.colab import files

for path in [
    "data/13_readout_policy_candidates.csv",
    "data/13_readout_policy_objectives.csv",
    "data/13_readout_policy_pareto_frontier.csv",
    "data/13_best_policy_by_weights.csv",
    "figures/figure_13a_accuracy_vs_queries_pareto.svg",
    "figures/figure_13b_coverage_vs_queries.svg",
    "figures/figure_13c_pareto_coverage_view.svg",
    "figures/figure_13d_best_policy_by_weights.svg",
    "figures/figure_13e_score_heatmap.svg",
]:
    files.download(path)
